# Inspect Steam match data

Read local **`data/steam/*.json.gz`** and **`*.json`** files using pandas. This notebook
loads the saved responses in memory and does not change files or call the Steam API.

Install dependencies with `python -m pip install -r notebooks/requirements.txt` from the
repository root, select that Python environment, then run cells in order. Works when
started from either the repository root or `notebooks/`.

The nested response becomes four tables: `matches`, `players`, `ability_upgrades`, and
`picks_bans`. Source filenames are retained so overlapping batches remain visible.


In [1]:
from pathlib import Path
import gzip
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks" / "requirements.txt").is_file()
             and (p / "dags" / "dota" / "steam.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook from the dota-pipeline repository.")
STEAM_DIR = ROOT / "data" / "steam"
FOCUS_MATCH_ID = None  # Set a match ID, or leave None to use the first loaded match.
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)


## 1. Discover and load saved responses

Each response is expected to contain `result.matches`. Failed responses and unexpected
structures raise an error naming the source file. `payloads` retains the original JSON.


In [2]:
files = sorted(set(STEAM_DIR.glob("*.json.gz")) | set(STEAM_DIR.glob("*.json")))
if not files:
    raise FileNotFoundError(f"No .json.gz or .json files found in {STEAM_DIR}")

payloads = {}
file_rows = []
for path in files:
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as stream:
        payload = json.load(stream)
    result = payload.get("result") if isinstance(payload, dict) else None
    if not isinstance(result, dict) or not isinstance(result.get("matches"), list):
        raise ValueError(f"Expected result.matches to be a list: {path}")
    if result.get("status", 1) != 1:
        raise ValueError(f"Steam response failed in {path}: {result.get('status')}")
    if any(not isinstance(match, dict) or "match_id" not in match
           for match in result["matches"]):
        raise ValueError(f"Expected match objects with match_id: {path}")
    payloads[path.name] = payload
    file_rows.append({"source_file": path.name, "size_bytes": path.stat().st_size,
                      "status": result.get("status"), "match_rows": len(result["matches"])})
file_summary = pd.DataFrame(file_rows)
display(file_summary)


,source_file,size_bytes,status,match_rows
0,seq_7624888547.json.gz,209916,1,100


## 2. Unpack matches and nested records

`matches` has one row per match occurrence in a file; `players` has one row per player
occurrence. Ability upgrades also carry `player_slot`. Picks/bans carry their match ID.
Join within a source using `source_file` plus `match_id` (and `player_slot` for players).
No records are deduplicated or filtered. Numeric IDs and raw timestamps stay as supplied.


In [3]:
match_rows, player_rows, ability_rows, draft_rows = [], [], [], []
for source_file, payload in payloads.items():
    for match in payload["result"]["matches"]:
        context = {"source_file": source_file, "match_id": match["match_id"]}
        match_rows.append({**{k: v for k, v in match.items()
                              if k not in {"players", "picks_bans"}}, **context})
        for player in match.get("players", []):
            player_rows.append({**{k: v for k, v in player.items()
                                   if k != "ability_upgrades"}, **context})
            for upgrade in player.get("ability_upgrades", []):
                ability_rows.append({**upgrade, **context,
                                     "player_slot": player.get("player_slot")})
        for pick_ban in match.get("picks_bans", []):
            draft_rows.append({**pick_ban, **context})

def frame_with_keys(rows, keys):
    return (pd.json_normalize(rows).convert_dtypes() if rows
            else pd.DataFrame(columns=keys))

matches = frame_with_keys(match_rows, ["source_file", "match_id"])
players = frame_with_keys(player_rows, ["source_file", "match_id", "player_slot"])
ability_upgrades = frame_with_keys(ability_rows, ["source_file", "match_id", "player_slot"])
picks_bans = frame_with_keys(draft_rows, ["source_file", "match_id"])

# Add convenience columns without replacing raw values.
if "start_time" in matches:
    matches["start_time_utc"] = pd.to_datetime(matches["start_time"], unit="s", utc=True)
if "duration" in matches:
    matches["duration_minutes"] = matches["duration"] / 60

tables = {"matches": matches, "players": players,
          "ability_upgrades": ability_upgrades, "picks_bans": picks_bans}
display(pd.DataFrame([{"table": name, "rows": len(df), "columns": len(df.columns)}
                      for name, df in tables.items()]))
for name, df in tables.items():
    print(name)
    display(df.head())


,table,rows,columns
0,matches,100,23
1,players,1000,41
2,ability_upgrades,18120,6
3,picks_bans,1241,6


matches


,radiant_win,duration,pre_game_duration,start_time,match_id,match_seq_num,tower_status_radiant,tower_status_dire,barracks_status_radiant,barracks_status_dire,cluster,first_blood_time,lobby_type,human_players,leagueid,game_mode,flags,engine,radiant_score,dire_score,source_file,start_time_utc,duration_minutes
0,True,2328,90,1794072429,9080432074,7624888547,1974,0,63,0,122,25,7,10,0,22,1,1,49,31,seq_7624888547.json.gz,2026-11-07 17:27:09+00:00,38.8
1,False,2052,60,1794072787,9080441042,7624888548,262,1828,51,63,181,54,0,10,0,23,1,1,31,55,seq_7624888547.json.gz,2026-11-07 17:33:07+00:00,34.2
2,True,1876,60,1794072963,9080445405,7624888549,1956,260,63,51,154,23,0,10,0,23,1,1,43,37,seq_7624888547.json.gz,2026-11-07 17:36:03+00:00,31.266667
3,False,237,90,1794074617,9080489370,7624888551,2047,2047,63,63,442,0,14,0,0,13,0,1,0,0,seq_7624888547.json.gz,2026-11-07 18:03:37+00:00,3.95
4,True,2366,60,1794072478,9080432972,7624888552,1924,0,63,1,181,181,0,10,0,23,1,1,63,54,seq_7624888547.json.gz,2026-11-07 17:27:58+00:00,39.433333


players


,account_id,player_slot,team_number,team_slot,hero_id,hero_variant,item_0,item_1,item_2,item_3,item_4,item_5,backpack_0,backpack_1,backpack_2,item_neutral,item_neutral2,kills,deaths,assists,...,last_hits,denies,gold_per_min,xp_per_min,level,net_worth,aghanims_scepter,aghanims_shard,moonshard,hero_damage,tower_damage,hero_healing,gold,gold_spent,scaled_hero_damage,scaled_tower_damage,scaled_hero_healing,source_file,match_id,additional_units
0,4294967295,0,0,0,27,0,232,1,180,108,1802,279,0,36,0,359,1595,5,6,16,...,143,6,428,555,20,15601,0,0,0,12706,7040,0,1416,15255,8898,2789,0,seq_7624888547.json.gz,9080432074,NaN
1,107429809,1,0,1,44,0,145,116,123,63,143,168,0,0,0,577,1584,10,5,11,...,288,6,604,774,23,22322,0,0,0,25612,16248,0,1697,21780,16582,7296,0,seq_7624888547.json.gz,9080432074,NaN
2,4294967295,2,0,2,101,0,180,244,36,57,232,206,23,0,0,1604,1577,4,7,19,...,27,1,297,439,17,10749,0,1,0,20285,383,0,839,9430,13553,201,0,seq_7624888547.json.gz,9080432074,NaN
3,4294967295,3,0,3,26,0,249,108,1,116,214,141,0,0,0,1864,1592,24,6,13,...,200,11,690,1238,27,24531,0,1,0,44176,4598,0,631,25815,34227,2839,0,seq_7624888547.json.gz,9080432074,NaN
4,89153575,4,0,4,28,0,116,1,939,73,63,178,0,0,0,1643,1586,6,7,13,...,172,5,486,752,23,17543,0,1,0,17368,3216,0,2133,16665,11426,1522,0,seq_7624888547.json.gz,9080432074,NaN


ability_upgrades


,ability,time,level,source_file,match_id,player_slot
0,5080,224,1,seq_7624888547.json.gz,9080432074,0
1,5078,376,2,seq_7624888547.json.gz,9080432074,0
2,5080,493,3,seq_7624888547.json.gz,9080432074,0
3,5078,605,4,seq_7624888547.json.gz,9080432074,0
4,5080,689,5,seq_7624888547.json.gz,9080432074,0


picks_bans


,is_pick,hero_id,team,order,source_file,match_id
0,True,37,1,0,seq_7624888547.json.gz,9080432074
1,True,27,0,1,seq_7624888547.json.gz,9080432074
2,True,86,1,2,seq_7624888547.json.gz,9080432074
3,True,26,0,3,seq_7624888547.json.gz,9080432074
4,True,104,1,4,seq_7624888547.json.gz,9080432074


## 3. Browse one match

Set `FOCUS_MATCH_ID` in setup and rerun to change matches. If a match appears in several
files, all occurrences are displayed with their source filename. Empty nested tables are
valid when the response does not contain those records.


In [4]:
if matches.empty:
    raise ValueError("The saved responses contain no matches.")
MATCH_ID = FOCUS_MATCH_ID if FOCUS_MATCH_ID is not None else int(matches.iloc[0]["match_id"])
if not matches["match_id"].eq(MATCH_ID).any():
    raise ValueError(f"Match {MATCH_ID} was not found. Choose an ID from matches['match_id'].")
print("Selected match:", MATCH_ID)
for name, df in tables.items():
    print(name)
    display(df.loc[df["match_id"].eq(MATCH_ID)])


Selected match: 9080432074
matches


,radiant_win,duration,pre_game_duration,start_time,match_id,match_seq_num,tower_status_radiant,tower_status_dire,barracks_status_radiant,barracks_status_dire,cluster,first_blood_time,lobby_type,human_players,leagueid,game_mode,flags,engine,radiant_score,dire_score,source_file,start_time_utc,duration_minutes
0,True,2328,90,1794072429,9080432074,7624888547,1974,0,63,0,122,25,7,10,0,22,1,1,49,31,seq_7624888547.json.gz,2026-11-07 17:27:09+00:00,38.8


players


,account_id,player_slot,team_number,team_slot,hero_id,hero_variant,item_0,item_1,item_2,item_3,item_4,item_5,backpack_0,backpack_1,backpack_2,item_neutral,item_neutral2,kills,deaths,assists,...,last_hits,denies,gold_per_min,xp_per_min,level,net_worth,aghanims_scepter,aghanims_shard,moonshard,hero_damage,tower_damage,hero_healing,gold,gold_spent,scaled_hero_damage,scaled_tower_damage,scaled_hero_healing,source_file,match_id,additional_units
0,4294967295,0,0,0,27,0,232,1,180,108,1802,279,0,36,0,359,1595,5,6,16,...,143,6,428,555,20,15601,0,0,0,12706,7040,0,1416,15255,8898,2789,0,seq_7624888547.json.gz,9080432074,NaN
1,107429809,1,0,1,44,0,145,116,123,63,143,168,0,0,0,577,1584,10,5,11,...,288,6,604,774,23,22322,0,0,0,25612,16248,0,1697,21780,16582,7296,0,seq_7624888547.json.gz,9080432074,NaN
2,4294967295,2,0,2,101,0,180,244,36,57,232,206,23,0,0,1604,1577,4,7,19,...,27,1,297,439,17,10749,0,1,0,20285,383,0,839,9430,13553,201,0,seq_7624888547.json.gz,9080432074,NaN
3,4294967295,3,0,3,26,0,249,108,1,116,214,141,0,0,0,1864,1592,24,6,13,...,200,11,690,1238,27,24531,0,1,0,44176,4598,0,631,25815,34227,2839,0,seq_7624888547.json.gz,9080432074,NaN
4,89153575,4,0,4,28,0,116,1,939,73,63,178,0,0,0,1643,1586,6,7,13,...,172,5,486,752,23,17543,0,1,0,17368,3216,0,2133,16665,11426,1522,0,seq_7624888547.json.gz,9080432074,NaN
5,4294967295,128,1,0,37,0,40,102,180,42,1802,108,34,279,0,1873,1593,2,8,14,...,167,0,420,534,19,14508,0,1,0,13573,685,6584,1428,14240,9582,471,3926,seq_7624888547.json.gz,9080432074,NaN
6,4294967295,129,1,1,97,0,256,265,116,63,57,1,40,0,0,1598,1583,11,9,13,...,107,10,406,607,20,12076,0,0,0,20705,0,0,209,15375,15480,0,0,seq_7624888547.json.gz,9080432074,NaN
7,158553888,130,1,2,86,0,180,0,1852,1,40,44,16,16,0,1604,1583,4,14,15,...,68,4,319,433,17,9419,0,1,0,17267,0,48,2194,8220,12072,0,14,seq_7624888547.json.gz,9080432074,NaN
8,4294967295,131,1,3,104,0,1,36,939,127,50,21,0,0,0,574,1586,8,11,14,...,168,11,437,742,23,14325,0,0,0,28421,430,3516,2065,13660,20136,266,2039,seq_7624888547.json.gz,9080432074,NaN
9,4294967295,132,1,4,1,0,147,145,63,34,75,30,0,0,0,1643,1584,4,7,5,...,252,9,417,542,19,14937,0,0,0,8045,962,0,3182,12050,5548,586,0,seq_7624888547.json.gz,9080432074,NaN


ability_upgrades


,ability,time,level,source_file,match_id,player_slot
0,5080,224,1,seq_7624888547.json.gz,9080432074,0
1,5078,376,2,seq_7624888547.json.gz,9080432074,0
2,5080,493,3,seq_7624888547.json.gz,9080432074,0
3,5078,605,4,seq_7624888547.json.gz,9080432074,0
4,5080,689,5,seq_7624888547.json.gz,9080432074,0
...,...,...,...,...,...,...
170,5006,1532,13,seq_7624888547.json.gz,9080432074,132
171,7314,1607,14,seq_7624888547.json.gz,9080432074,132
172,7314,1681,15,seq_7624888547.json.gz,9080432074,132
173,665,1839,16,seq_7624888547.json.gz,9080432074,132


picks_bans


,is_pick,hero_id,team,order,source_file,match_id
0,True,37,1,0,seq_7624888547.json.gz,9080432074
1,True,27,0,1,seq_7624888547.json.gz,9080432074
2,True,86,1,2,seq_7624888547.json.gz,9080432074
3,True,26,0,3,seq_7624888547.json.gz,9080432074
4,True,104,1,4,seq_7624888547.json.gz,9080432074
5,True,28,0,5,seq_7624888547.json.gz,9080432074
6,True,101,0,6,seq_7624888547.json.gz,9080432074
7,True,97,1,7,seq_7624888547.json.gz,9080432074
8,True,44,0,8,seq_7624888547.json.gz,9080432074
9,True,1,1,9,seq_7624888547.json.gz,9080432074


## 4. Inspect columns, missing values, and overlapping batches

Repeated match IDs can occur when saved sequence batches overlap. Counts below describe
what was loaded; they do not remove duplicates or apply cleaning rules. Use `SCHEMA_TABLE`
to inspect another table. The player-count report shows only matches present in `players`.


In [5]:
SCHEMA_TABLE = "players"  # matches, players, ability_upgrades, or picks_bans
selected = tables[SCHEMA_TABLE]
display(pd.DataFrame({"dtype": selected.dtypes.astype(str),
                      "missing_rows": selected.isna().sum()}).rename_axis("column"))

match_counts = matches.groupby("match_id", dropna=False).size().rename("occurrences")
print("Match IDs appearing more than once:")
display(match_counts.loc[match_counts.gt(1)].reset_index())
print("Player rows by source and match:")
display(players.groupby(["source_file", "match_id"], dropna=False)
        .size().rename("player_rows").reset_index())


,dtype,missing_rows
column,,
account_id,Int64,37
player_slot,Int64,0
team_number,Int64,0
team_slot,Int64,0
hero_id,Int64,0
hero_variant,Int64,0
item_0,Int64,0
item_1,Int64,0
item_2,Int64,0


Match IDs appearing more than once:


,match_id,occurrences


Player rows by source and match:


,source_file,match_id,player_rows
0,seq_7624888547.json.gz,9080363565,10
1,seq_7624888547.json.gz,9080395694,10
2,seq_7624888547.json.gz,9080397369,10
3,seq_7624888547.json.gz,9080402803,10
4,seq_7624888547.json.gz,9080404428,10
...,...,...,...
95,seq_7624888547.json.gz,9080468874,10
96,seq_7624888547.json.gz,9080469331,10
97,seq_7624888547.json.gz,9080472097,10
98,seq_7624888547.json.gz,9080484689,10


## 5. Inspect the original JSON

The tables omit nested lists from their parent rows to make browsing easier. The original
response remains available in `payloads` for inspecting exact fields and nested structures.


In [6]:
SOURCE_FILE = files[0].name
raw_match = next((match for match in payloads[SOURCE_FILE]["result"]["matches"]
                  if match["match_id"] == MATCH_ID), None)
if raw_match is None:
    print(f"Match {MATCH_ID} is not in {SOURCE_FILE}; select another SOURCE_FILE.")
else:
    print(json.dumps(raw_match, indent=2))


{
  "players": [
    {
      "account_id": 4294967295,
      "player_slot": 0,
      "team_number": 0,
      "team_slot": 0,
      "hero_id": 27,
      "hero_variant": 0,
      "item_0": 232,
      "item_1": 1,
      "item_2": 180,
      "item_3": 108,
      "item_4": 1802,
      "item_5": 279,
      "backpack_0": 0,
      "backpack_1": 36,
      "backpack_2": 0,
      "item_neutral": 359,
      "item_neutral2": 1595,
      "kills": 5,
      "deaths": 6,
      "assists": 16,
      "leaver_status": 0,
      "last_hits": 143,
      "denies": 6,
      "gold_per_min": 428,
      "xp_per_min": 555,
      "level": 20,
      "net_worth": 15601,
      "aghanims_scepter": 0,
      "aghanims_shard": 0,
      "moonshard": 0,
      "hero_damage": 12706,
      "tower_damage": 7040,
      "hero_healing": 0,
      "gold": 1416,
      "gold_spent": 15255,
      "scaled_hero_damage": 8898,
      "scaled_tower_damage": 2789,
      "scaled_hero_healing": 0,
      "ability_upgrades": [
        {
         